# 自定义中间件：用一个天气查询实例学习

对应课件第 5 节“自定义中间件”，只学习自定义 hook。

所有示例都围绕同一个问题：**查询杭州今天的天气**。
工具返回固定的教学模拟数据，重点观察中间件什么时候执行、修改了什么。

建议先学第 1 个示例，再对照第 2 个示例看类写法；理解后再学 Wrap-style。

| 风格 | 钩子 | 一句话理解 |
| --- | --- | --- |
| Node-style | `before_agent` | 整次 Agent 调用开始前 |
| Node-style | `before_model` | 每次模型调用前 |
| Node-style | `after_model` | 每次模型调用后 |
| Node-style | `after_agent` | 整次 Agent 调用结束后 |
| Wrap-style | `wrap_model_call` | 把一次模型调用包起来 |
| Wrap-style | `wrap_tool_call` | 把一次工具调用包起来 |

**运行方法**：选择项目的 `langchain` Python 内核，先运行“公共准备”，再运行要学习的示例。
示例沿用根目录 `.env` 中的 `BAILIAN_API_KEY`、`BAILIAN_API_URL` 和已有模型名。
运行各示例的 `agent.invoke(...)` 会请求你配置的模型；天气工具本身不访问网络。

## 0、公共准备

与现有工程保持一致：读取 `.env`，然后创建 `model`。
本单元只初始化模型，不发起对话请求。

In [1]:
import os
from typing import Any, Callable

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware, AgentState, ModelRequest, ModelResponse,
    before_agent, before_model, after_model, after_agent,
    wrap_model_call, wrap_tool_call,
)
from langchain.tools import tool
from langchain.tools.tool_node import ToolCallRequest
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.runtime import Runtime
from langgraph.types import Command

load_dotenv(override = True)

BAILIAN_API_KEY = os.getenv("BAILIAN_API_KEY")
BAILIAN_API_URL = os.getenv("BAILIAN_API_URL")

model = init_chat_model(
    model = "openai:qwen3.8-max",
    api_key = BAILIAN_API_KEY,
    base_url = BAILIAN_API_URL,
)

In [2]:
@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """查询指定城市的天气，返回固定的教学模拟数据。

    Args:
        city: 城市名称，例如杭州。
        is_forcast: 是否同时返回明天的天气预报。
    """
    print(f"  [工具实际执行] city={city}, is_forcast={is_forcast}")
    res = f"[教学模拟] {city}今天晴，25℃。"
    if is_forcast:
        res += "明天小雨，22℃。"
    return res


# 沿用课件的 is_forcast 命名，方便与原例对照。
system_prompt = (
    "你是天气查询助手。天气问题必须使用 get_weather 工具。"
    "一次查询只调用一次工具，拿到结果后直接回答，并注明数据是教学模拟。"
)
question = "请使用 get_weather 查询杭州今天的天气，不需要明天的预报。"

## 1、Node-style：基于装饰器

先把 hook 当成“到某个环节就自动执行的函数”。
这里的四个函数只打印日志，所以都返回 `None`。

- `state`：当前 Agent 状态，`state["messages"]` 是消息列表。
- `runtime`：框架传入的运行上下文；本例暂时不用。
- 列表里的消息是消息对象，因此用 `.content`，不用 `["content"]`。

In [3]:
@before_agent
def before_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    print("[before_agent] 开始处理用户问题")
    return None


@before_model
def before_model_middleware(state: AgentState, runtime: Runtime) -> None:
    print(f"[before_model] 即将调用模型，当前有 {len(state['messages'])} 条消息")
    return None


@after_model
def after_model_middleware(state: AgentState, runtime: Runtime) -> None:
    last_message = state["messages"][-1]
    print(f"[after_model] 模型调用完成，发出了 {len(last_message.tool_calls)} 个工具请求")
    return None


@after_agent
def after_agent_middleware(state: AgentState, runtime: Runtime) -> None:
    print("[after_agent] 整个 Agent 流程结束")
    return None


node_decorator_agent = create_agent(
    model = model,
    tools = [get_weather],
    system_prompt = system_prompt,
    middleware = [
        before_agent_middleware,
        before_model_middleware,
        after_model_middleware,
        after_agent_middleware,
    ],
)

node_decorator_response = node_decorator_agent.invoke({
    "messages": [HumanMessage(question)],
})
node_decorator_response["messages"][-1].pretty_print()

[before_agent] 开始处理用户问题
[before_model] 即将调用模型，当前有 1 条消息
[after_model] 模型调用完成，发出了 1 个工具请求
  [工具实际执行] city=杭州, is_forcast=False
[before_model] 即将调用模型，当前有 3 条消息
[after_model] 模型调用完成，发出了 0 个工具请求
[after_agent] 整个 Agent 流程结束
================================== Ai Message ==================================

杭州今天晴，气温 25℃。

（注：以上数据为教学模拟数据，非真实天气信息。）


### 为什么 `before_model` 和 `after_model` 通常各执行两次？

本例预期的调用路径是：

```text
before_agent
  before_model → 模型决定调用天气工具 → after_model
  get_weather 实际执行
  before_model → 模型读取工具结果并回答 → after_model
after_agent
```

**一次 `agent.invoke()` 里面可以包含多次模型调用。**
`before_agent` / `after_agent` 只围绕整次调用，
`before_model` / `after_model` 则围绕每次模型调用。
如果真实模型选择直接回答或重复调用工具，日志次数也会随之变化。

## 2、Node-style：基于类

和示例 1 做同一件事，只是把四个 hook 放进同一个类。
类名可以自定，必须继承 `AgentMiddleware`，方法名必须与框架约定一致。

In [4]:
class NodeStyleMiddleware(AgentMiddleware):
    def before_agent(self, state: AgentState, runtime: Runtime) -> None:
        print("[类 before_agent] 开始处理用户问题")
        return None

    def before_model(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[类 before_model] 即将调用模型，当前有 {len(state['messages'])} 条消息")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> None:
        last_message = state["messages"][-1]
        print(f"[类 after_model] 模型调用完成，发出了 {len(last_message.tool_calls)} 个工具请求")
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> None:
        print("[类 after_agent] 整个 Agent 流程结束")
        return None


node_class_agent = create_agent(
    model = model,
    tools = [get_weather],
    system_prompt = system_prompt,
    middleware = [NodeStyleMiddleware()],
)

node_class_response = node_class_agent.invoke({
    "messages": [HumanMessage(question)],
})
node_class_response["messages"][-1].pretty_print()

[类 before_agent] 开始处理用户问题
[类 before_model] 即将调用模型，当前有 1 条消息
[类 after_model] 模型调用完成，发出了 1 个工具请求
  [工具实际执行] city=杭州, is_forcast=False
[类 before_model] 即将调用模型，当前有 3 条消息
[类 after_model] 模型调用完成，发出了 0 个工具请求
[类 after_agent] 整个 Agent 流程结束
================================== Ai Message ==================================

杭州今天的天气：晴，气温 25℃。

说明：以上数据来自教学模拟环境（get_weather 工具返回的固定示例数据），并非真实天气信息，仅供参考。


## 3、Wrap-style：基于装饰器

最关键的是 **`handler(request)`**：执行被包装的那次调用，拿回结果。

```text
handler 前的代码 → 实际调用 → handler 后的代码
```

- `wrap_model_call`：在模型调用前后打印日志。
- `wrap_tool_call`：先把 `is_forcast` 改为 `True`，再执行工具。

用户只问今天，但中间件会让工具同时返回明天的预报。
注意日志中的“原始参数”和“修改后参数”，这就是中间件对流程的干预。

课件为比较前后效果调用了两次工具；这里先改参数、再调用一次，更容易观察。
`request.override(...)` 创建修改后的请求，保留原工具调用的名称、ID 等信息。

In [5]:
@wrap_model_call
def wrap_model_call_middleware(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    print("[wrap_model_call] 模型调用前")
    response = handler(request)
    print("[wrap_model_call] 模型调用后")
    return response


@wrap_tool_call
def wrap_tool_call_middleware(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    print(f"[wrap_tool_call] 原始参数：{request.tool_call['args']}")
    new_args = {**request.tool_call["args"], "is_forcast": True}
    modified_request = request.override(
        tool_call = {**request.tool_call, "args": new_args},
    )
    print(f"[wrap_tool_call] 修改后参数：{new_args}")
    result = handler(modified_request)
    print("[wrap_tool_call] 工具调用完成")
    return result


wrap_decorator_agent = create_agent(
    model = model,
    tools = [get_weather],
    system_prompt = system_prompt,
    middleware = [wrap_model_call_middleware, wrap_tool_call_middleware],
)

wrap_decorator_response = wrap_decorator_agent.invoke({
    "messages": [HumanMessage(question)],
})
# 直接查看 ToolMessage，确认工具结果里已经有明天的预报。
for msg in wrap_decorator_response["messages"]:
    msg.pretty_print()

[wrap_model_call] 模型调用前
[wrap_model_call] 模型调用后
[wrap_tool_call] 原始参数：{'city': '杭州', 'is_forcast': False}
[wrap_tool_call] 修改后参数：{'city': '杭州', 'is_forcast': True}
  [工具实际执行] city=杭州, is_forcast=True
[wrap_tool_call] 工具调用完成
[wrap_model_call] 模型调用前
[wrap_model_call] 模型调用后
================================ Human Message =================================

请使用 get_weather 查询杭州今天的天气，不需要明天的预报。
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_b0fd0c46c7c346caba1c3800)
 Call ID: call_b0fd0c46c7c346caba1c3800
  Args:
    city: 杭州
    is_forcast: False
================================= Tool Message =================================
Name: get_weather

[教学模拟] 杭州今天晴，25℃。明天小雨，22℃。
================================== Ai Message ==================================

杭州今天晴，气温 25℃。

（以上数据为教学模拟数据，仅供参考。）


## 4、Wrap-style：基于类

和示例 3 的行为一致，把两个包装钩子集中到一个类中。
仍然记住：调用 `handler(...)` 才会进入下一层，并把结果 `return` 回去。

In [6]:
class WrapStyleMiddleware(AgentMiddleware):
    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        print("[类 wrap_model_call] 模型调用前")
        response = handler(request)
        print("[类 wrap_model_call] 模型调用后")
        return response

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        print(f"[类 wrap_tool_call] 原始参数：{request.tool_call['args']}")
        new_args = {**request.tool_call["args"], "is_forcast": True}
        modified_request = request.override(
            tool_call = {**request.tool_call, "args": new_args},
        )
        print(f"[类 wrap_tool_call] 修改后参数：{new_args}")
        result = handler(modified_request)
        print("[类 wrap_tool_call] 工具调用完成")
        return result


wrap_class_agent = create_agent(
    model = model,
    tools = [get_weather],
    system_prompt = system_prompt,
    middleware = [WrapStyleMiddleware()],
)

wrap_class_response = wrap_class_agent.invoke({
    "messages": [HumanMessage(question)],
})
for msg in wrap_class_response["messages"]:
    msg.pretty_print()

[类 wrap_model_call] 模型调用前
[类 wrap_model_call] 模型调用后
[类 wrap_tool_call] 原始参数：{'city': '杭州', 'is_forcast': False}
[类 wrap_tool_call] 修改后参数：{'city': '杭州', 'is_forcast': True}
  [工具实际执行] city=杭州, is_forcast=True
[类 wrap_tool_call] 工具调用完成
[类 wrap_model_call] 模型调用前
[类 wrap_model_call] 模型调用后
================================ Human Message =================================

请使用 get_weather 查询杭州今天的天气，不需要明天的预报。
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_af1ae702d9e543edacf1a555)
 Call ID: call_af1ae702d9e543edacf1a555
  Args:
    city: 杭州
    is_forcast: False
================================= Tool Message =================================
Name: get_weather

[教学模拟] 杭州今天晴，25℃。明天小雨，22℃。
================================== Ai Message ==================================

杭州今天的天气：**晴，气温 25℃**。

您只需要今天的天气，所以明天那部分（小雨，22℃）我就不展开了，如需查看请告诉我。

以上数据为教学模拟数据，并非实时真实天气。


## 5、补充：`can_jump_to` 提前结束

前四个示例理解后再看这里。用户说“不查了”时，在调用模型前直接结束。

- `can_jump_to=["end"]`：先声明这个 hook 允许跳到结尾。
- `"jump_to": "end"`：本次真正执行跳转。
- `"messages": [AIMessage(...)]`：向状态追加一条结束消息。

本例使用框架支持的 `"end"`，与装饰器的声明保持一致。
如果还注册了 `after_agent`，提前结束时仍会进入该结束钩子。

In [7]:
@before_model(can_jump_to=["end"])
def stop_query_middleware(
    state: AgentState, runtime: Runtime,
) -> dict[str, Any] | None:
    if "不查了" in state["messages"][-1].content:
        print("[提前结束] 跳过模型和工具调用")
        return {
            "messages": [AIMessage(content="已停止查询。")],
            "jump_to": "end",
        }
    return None


stop_agent = create_agent(
    model = model,
    tools = [get_weather],
    system_prompt = system_prompt,
    middleware = [stop_query_middleware],
)

stop_response = stop_agent.invoke({
    "messages": [HumanMessage("不查了，请结束。")],
})
stop_response["messages"][-1].pretty_print()

[提前结束] 跳过模型和工具调用
================================== Ai Message ==================================

已停止查询。


## 6、补充：多个中间件的执行顺序

用 A、B 两个中间件观察课件的“洋葱结构”。
此例不提供天气工具，只让模型打招呼，方便观察一轮模型调用。

同一个类创建多个实例时，给它们不同的 `name`，让框架能够区分。
执行顺序由 `middleware=[...]` 的排列决定，与类写在哪里无关。

In [ ]:
class OrderMiddleware(AgentMiddleware):
    def __init__(self, label: str):
        self.label = label

    @property
    def name(self) -> str:
        return self.label

    def before_model(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[{self.label}] before_model")
        return None

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        print(f"[{self.label}] wrap_model_call 进入")
        response = handler(request)
        print(f"[{self.label}] wrap_model_call 返回")
        return response

    def after_model(self, state: AgentState, runtime: Runtime) -> None:
        print(f"[{self.label}] after_model")
        return None


order_agent = create_agent(
    model = model,
    middleware = [OrderMiddleware("A"), OrderMiddleware("B")],
)
order_response = order_agent.invoke({
    "messages": [HumanMessage("你好，请打个招呼。")],
})
order_response["messages"][-1].pretty_print()

预期日志顺序：

```text
[A] before_model
[B] before_model
[A] wrap_model_call 进入
[B] wrap_model_call 进入
     模型调用
[B] wrap_model_call 返回
[A] wrap_model_call 返回
[B] after_model
[A] after_model
```

### 动手改三处

1. 示例 3 把 `"is_forcast": True` 改成 `False`，观察 ToolMessage 是否还包含明天的天气。
2. 示例 5 把“不查了，请结束。”改成 `question`，观察正常问题是否继续执行。
3. 示例 6 对调 A、B 的传入顺序，观察进入和返回的日志顺序。

### 本例需要记住的区别

- Node-style：在指定节点运行；返回 `None` 表示本 hook 不提交状态更新，返回字典则提交更新。
- Wrap-style：通过 `handler(request)` 包装调用；本例先改工具请求，再执行一次工具。
- 单个简单 hook 可以用装饰器；需要组织多个 hook 时，类写法更集中。
- `before_*` 正序，`after_*` 逆序，`wrap_*` 先传入的包在外层。

对应课件：5.2、5.3、5.4、5.5、5.6。
接口与执行顺序参考：[LangChain 官方自定义中间件文档](https://docs.langchain.com/oss/python/langchain/middleware/custom)。